In [ ]:
# Resolve the repository root so data/ and output/ paths work from any working directory
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Start

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import statsmodels.api as sm
import linearmodels.panel as plm
import statsmodels.formula.api as smf
import os

In [ ]:
output_dir = str(ROOT / "output/Inequality")
os.makedirs(output_dir, exist_ok=True)


In this notebook the goal is estimating the Regional Inequality using the three main data set

# Reading the data

In [ ]:
df = pd.read_excel(
    str(ROOT / "data/Validation/hies_ntl_gdp_master.xlsx")
)

# Compute Gini inequality index for three variable (NTL, GDP, Expenditure)

first of all I have to define the inequality indexes (GE, Gini, CV)

In [ ]:
# Generalized Entropy family (population-weighted)
# Following Lessmann & Seidel (2017):
#   alpha = -1  -> GE(-1)
#   alpha =  0  -> Mean Log Deviation (MLD)
#   alpha =  1  -> Theil index
#   CV and Gini computed separately (not part of the GE family formula)

def weighted_GE(values, weights, alpha):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = ~(np.isnan(values) | np.isnan(weights)) & (values > 0)
    values = values[mask]
    weights = weights[mask]

    if len(values) < 2:
        return np.nan

    p = weights / weights.sum()
    y_bar = np.sum(p * values)
    ratio = values / y_bar

    if alpha == 0:
        # Mean Log Deviation
        ge = np.sum(p * (-np.log(ratio)))
    elif alpha == 1:
        # Theil index
        ge = np.sum(p * ratio * np.log(ratio))
    else:
        # General case (e.g. alpha = -1)
        ge = (1 / (alpha * (alpha - 1))) * np.sum(p * (ratio**alpha - 1))

    return ge


def weighted_CV(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = ~(np.isnan(values) | np.isnan(weights))
    values = values[mask]
    weights = weights[mask]

    if len(values) < 2:
        return np.nan

    p = weights / weights.sum()
    y_bar = np.sum(p * values)
    variance = np.sum(p * (values - y_bar) ** 2)
    return np.sqrt(variance) / y_bar


def weighted_gini(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = ~(np.isnan(values) | np.isnan(weights))
    values = values[mask]
    weights = weights[mask]

    if len(values) < 2:
        return np.nan

    p = weights / weights.sum()
    y_bar = np.sum(p * values)

    diff_matrix = np.abs(values[:, None] - values[None, :])
    weight_matrix = np.outer(p, p)

    return np.sum(weight_matrix * diff_matrix) / (2 * y_bar)

now we compute gini for each

In [ ]:
variables = {
    "NTL_Per_Capita": "NTL",
    "GDP_per_Capita": "GDP",
    "real_exp_per_capita": "Exp",
}

records = []
for year, group in df.groupby("Year"):
    row = {"Year": year}
    for var, label in variables.items():
        w = group["Population"]
        v = group[var]
        row[f"Gini_{label}"]  = weighted_gini(v, w)
        row[f"GEm1_{label}"]  = weighted_GE(v, w, alpha=-1)
        row[f"MLD_{label}"]   = weighted_GE(v, w, alpha=0)
        row[f"Theil_{label}"] = weighted_GE(v, w, alpha=1)
        row[f"CV_{label}"]    = weighted_CV(v, w)
    records.append(row)

ineq_df = pd.DataFrame(records).sort_values("Year").reset_index(drop=True)
print(ineq_df)
ineq_df.to_excel(os.path.join(output_dir, "Inequality_measures_all.xlsx"), index=False)

In [ ]:
measures = ["Gini", "GEm1", "MLD", "Theil", "CV"]
titles = {
    "Gini": "Population-weighted Gini Coefficient",
    "GEm1": "Generalized Entropy GE(-1)",
    "MLD": "Mean Log Deviation (GE(0))",
    "Theil": "Theil Index (GE(1))",
    "CV": "Coefficient of Variation",
}

sanction_years = [(1391, "2012 sanctions"), (1397, "2018 JCPOA exit")]

for m in measures:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for label, marker in zip(["NTL", "GDP", "Exp"], ["o", "s", "^"]):
        col = f"{m}_{label}"
        ax.plot(ineq_df["Year"], ineq_df[col], marker=marker, label=label)

    for shock_year, txt in sanction_years:
        if shock_year in ineq_df["Year"].values:
            ax.axvline(shock_year, color="gray", linestyle="--", alpha=0.5)
            ax.text(shock_year, ax.get_ylim()[1]*0.97, txt, rotation=90,
                    va="top", ha="right", fontsize=8, color="gray")

    ax.set_xlabel("Year (Shamsi)")
    ax.set_ylabel(titles[m])
    ax.set_title(f"Spatial Inequality Trend — {titles[m]}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Inequality_{m}_trend.png"), dpi=300)
    plt.show()

# Structural Break test

The 2012/1391 shock dummy is deliberately omitted. With the sample starting in 1391, a level/slope dummy for 1391 is either collinear with the intercept (if defined from 1391) or identified off a single pre-shock observation (if defined from 1392) neither is econometrically defensible. The analysis therefore focuses on the identifiable 2018/1397 shock only.

In [ ]:
def run_structural_break(df, measure_col, shock_year=1397):
    sub = df[["Year", measure_col]].dropna().sort_values("Year").reset_index(drop=True)
    if len(sub) < 5:
        return None

    sub["Trend"] = sub["Year"] - sub["Year"].min()
    sub["D2018"] = (sub["Year"] >= shock_year).astype(int)
    sub["D2018_Trend"] = sub["D2018"] * sub["Trend"]

    X = sm.add_constant(sub[["Trend", "D2018", "D2018_Trend"]])
    y = sub[measure_col]

    model = sm.OLS(y, X).fit()
    return model, sub

In [ ]:
results_summary = []

for var in ["NTL", "GDP", "Exp"]:
    for m in measures:  # reusing the `measures` list already defined above
        col = f"{m}_{var}"
        if col not in ineq_df.columns:
            continue

        result = run_structural_break(ineq_df, col)
        if result is None:
            continue
        model, sub = result

        results_summary.append({
            "Variable": var,
            "Measure": m,
            "N": int(model.nobs),
            "Pre_Trend_b1": round(model.params.get("Trend", np.nan), 5),
            "Level_Shift_b2": round(model.params.get("D2018", np.nan), 5),
            "Slope_Change_b3": round(model.params.get("D2018_Trend", np.nan), 5),
            "SE_Slope_Change": round(model.bse.get("D2018_Trend", np.nan), 5),
            "p_value_Slope_Change": round(model.pvalues.get("D2018_Trend", np.nan), 4),
            "R2": round(model.rsquared, 3),
        })

summary_df = pd.DataFrame(results_summary)
print(summary_df)

summary_df.to_excel(os.path.join(output_dir, "..", "StructuralBreak_summary.xlsx"), index=False)


In [ ]:
with open(os.path.join(output_dir, "..", "StructuralBreak_full_output.txt"), "w", encoding="utf-8") as f:
    for var in ["NTL", "GDP", "Exp"]:
        for m in measures:
            col = f"{m}_{var}"
            if col not in ineq_df.columns:
                continue
            result = run_structural_break(ineq_df, col)
            if result is None:
                continue
            model, sub = result
            f.write(f"===== {col} (N={int(model.nobs)}) =====\n")
            f.write(model.summary().as_text())
            f.write("\n\n")

# Robustness check (Gas flare)

Recompute all five inequality measures AND the structural break test after dropping the four low-population oil provinces whose per-capita NTL is contaminated by gas flaring.

In [ ]:
flare_provinces = ["Bushehr", "Khuzestan", "Ilam", "Kohgiluyeh & Boyerahmad"]

# --- adapt the province column name if yours differs ---
prov_col = "Province_EN" if "Province_EN" in df.columns else "Province"

# sanity check: make sure the names actually match the data
present = set(df[prov_col].unique())
missing = [p for p in flare_provinces if p not in present]
if missing:
    print("WARNING - these flare names were NOT found in the data, check spelling:")
    print(missing)
    print("Available provinces:", sorted(present))

df_nf = df[~df[prov_col].isin(flare_provinces)].copy()
print(f"Full sample provinces: {df[prov_col].nunique()}  |  Non-flare sample: {df_nf[prov_col].nunique()}")


In [ ]:
records_nf = []
for year, group in df_nf.groupby("Year"):
    row = {"Year": year}
    for var, label in variables.items():   # `variables` already defined earlier
        w = group["Population"]
        v = group[var]
        row[f"Gini_{label}"]  = weighted_gini(v, w)
        row[f"GEm1_{label}"]  = weighted_GE(v, w, alpha=-1)
        row[f"MLD_{label}"]   = weighted_GE(v, w, alpha=0)
        row[f"Theil_{label}"] = weighted_GE(v, w, alpha=1)
        row[f"CV_{label}"]    = weighted_CV(v, w)
    records_nf.append(row)

ineq_df_nf = pd.DataFrame(records_nf).sort_values("Year").reset_index(drop=True)
print(ineq_df_nf)
ineq_df_nf.to_excel(os.path.join(output_dir, "Inequality_measures_nonflare.xlsx"), index=False)


In [ ]:
for m in measures:   # `measures` and `titles` already defined earlier
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for label, marker in zip(["NTL", "GDP", "Exp"], ["o", "s", "^"]):
        col = f"{m}_{label}"
        ax.plot(ineq_df_nf["Year"], ineq_df_nf[col], marker=marker, label=label)

    for shock_year, txt in sanction_years:
        if shock_year in ineq_df_nf["Year"].values:
            ax.axvline(shock_year, color="gray", linestyle="--", alpha=0.5)
            ax.text(shock_year, ax.get_ylim()[1]*0.97, txt, rotation=90,
                    va="top", ha="right", fontsize=8, color="gray")

    ax.set_xlabel("Year (Shamsi)")
    ax.set_ylabel(titles[m])
    ax.set_title(f"Spatial Inequality Trend (excl. flare provinces) — {titles[m]}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Inequality_{m}_trend_nonflare.png"), dpi=300)
    plt.show()

In [ ]:
results_summary_nf = []

for var in ["NTL", "GDP", "Exp"]:
    for m in measures:
        col = f"{m}_{var}"
        if col not in ineq_df_nf.columns:
            continue
        result = run_structural_break(ineq_df_nf, col)
        if result is None:
            continue
        model, sub = result
        results_summary_nf.append({
            "Variable": var,
            "Measure": m,
            "N": int(model.nobs),
            "Slope_Change_b3": round(model.params.get("D2018_Trend", np.nan), 5),
            "SE_Slope_Change": round(model.bse.get("D2018_Trend", np.nan), 5),
            "p_value_Slope_Change": round(model.pvalues.get("D2018_Trend", np.nan), 4),
            "R2": round(model.rsquared, 3),
        })

summary_df_nf = pd.DataFrame(results_summary_nf)
print(summary_df_nf)
summary_df_nf.to_excel(os.path.join(output_dir, "..", "StructuralBreak_summary_nonflare.xlsx"), index=False)
